In [1]:
import pandas as pd

# Auto download titanic.csv
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
pd.read_csv(url).to_csv('titanic.csv', index=False)

# Create student_marks_messy.csv
df = pd.DataFrame({
'Student_ID':[1,2,3,4,5,5,6,7,8,9,10,10,11,12],
'Name':['Anu','Rahul','Priya','Amit','Sara','Sara','John','Meena','Vikram','Neha','Arjun','Arjun','Adil','Anjali'],
'Gender':['M','F','Female','Male','f','f','MALE','FEMALE','m','F','M','M','m','FEMALE'],
'english_marks':['85','92','78%','88','absent','absent','90','85','82','79%','91','91','88%','95'],
'maths_marks':[80,88,75,90,82,82,None,95,70,85,89,89,78,92],
'science_marks':[85,92,80,85,88,88,90,92,75,80,85,85,82,96]
})
df.to_csv('student_marks_messy.csv', index=False)
print("Both files ready!")

Both files ready!


In [2]:
import pandas as pd
import numpy as np

# --- Load files ---
# upload both csvs in same folder as notebook
titanic = pd.read_csv('titanic.csv')
student = pd.read_csv('student_marks_messy.csv')

# ==========================================
# 1a — The Data Vault
# ==========================================
print("=== TITANIC ===")
print(titanic.shape)
print(titanic.info())
print(titanic.describe())
print(titanic.isnull().sum())

print("\n=== STUDENT MARKS ===")
print(student.shape)
print(student.info())
print(student.describe())
print(student.isnull().sum())

# ==========================================
# 1b — The Passenger Records
# ==========================================
# Every passenger above age 60 using.loc
above_60 = titanic.loc[titanic['Age'] > 60]
print(above_60.head())

# first 5 rows and 4 columns using.iloc
first_5x4 = titanic.iloc[:5, :4]
print(first_5x4)

# Explanation:
#.loc is label-based -> we filter by condition Age > 60
#.iloc is position-based -> we filter by index numbers 0:5, 0:4

# ==========================================
# 1c — The Corrupted File
# ==========================================
print("Before cleanup:", student.shape)
print(student['gender'].unique() if 'gender' in student.columns else student['Gender'].unique())

# Standardize column names first (lowercase)
student.columns = student.columns.str.strip().str.lower()

# 1. Fix gender column - handle M, m, Male, MALE, F, f, Female, etc
student['gender'] = student['gender'].str.strip().str.lower()
gender_map = {'m':'Male', 'male':'Male', 'f':'Female', 'female':'Female'}
student['gender'] = student['gender'].map(gender_map)

# 2. Correct english_marks to numeric
# It may have strings like '85%', 'absent', etc.
student['english_marks'] = pd.to_numeric(student['english_marks'], errors='coerce')

# 3. Remove duplicates
student = student.drop_duplicates()

print("After cleanup:", student.shape)
print(student['gender'].unique())

# ==========================================
# 1d — The Missing Pieces
# ==========================================
# Recover missing marks - use median of subject
for col in ['english_marks', 'maths_marks', 'science_marks']:
    if col in student.columns:
        student[col] = student[col].fillna(student[col].median())

# Repair missing Titanic Age using median grouped by Pclass
titanic['Age'] = titanic['Age'].fillna(
    titanic.groupby('Pclass')['Age'].transform('median')
)

print("Missing after fill - Student:\n", student.isnull().sum())
print("Missing after fill - Titanic Age:", titanic['Age'].isnull().sum())

# ==========================================
# 1e — The Hidden Pattern
# ==========================================
grouped = titanic.groupby('Pclass').agg(
    mean_fare=('Fare','mean'),
    mean_age=('Age','mean'),
    survival_rate=('Survived','mean') # 1 = survived, so mean = rate
).reset_index()

print(grouped)
# Story: Higher class = Higher fare, slightly higher age, much higher survival

# ==========================================
# 1f — The Port Connection
# ==========================================
# Lookup table for Embarked codes
port_lookup = pd.DataFrame({
    'Embarked': ['C', 'Q', 'S'],
    'Port_Name': ['Cherbourg', 'Queenstown', 'Southampton'],
    'Full_Name': ['Cherbourg, France', 'Queenstown, Ireland', 'Southampton, England']
})

# Merge with titanic to restore context
titanic_merged = pd.merge(titanic, port_lookup, on='Embarked', how='left')

print(titanic_merged[['Embarked','Port_Name','Full_Name']].head(10))
print(titanic_merged['Port_Name'].value_counts())

=== TITANIC ===
(891, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None
       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000    0.383838    2.308642   29.699118    0.523008   
std